In [55]:
import sys
from pathlib import Path
import os

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    # Strip ~/notebooks/ccfraud from PYTHON_PATH if notebook started in one of these subdirectories
    if root_dir.parts[-1:] == ('airquality',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
if os.path.exists(f"{root_dir}/.env"):
    settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/yuxinjin/Desktop/KTH/Second-P2/SML/Lab1
HopsworksSettings initialized!


<span style="font-width:bold; font-size: 3rem; color:#333;">- Part 02: Daily Feature Pipeline for Exchange Rate, Inflation, and Interest Rate</span>

## 🗒️ This notebook is divided into the following sections:
1. Retrieve Latest Exchange Rate Data
2. Retrieve Latest Inflation Data (Monthly)
3. Retrieve Latest Interest Rate Data
4. Feature Group Insertion


__This notebook should be scheduled to run daily__

**Note:** 
- Exchange rate data is updated daily
- Inflation data is monthly (only updated when new month's data is available)
- Interest rate data may be daily or monthly depending on your data source

You can use any Python Orchestration tool (GitHub Actions, Airflow, etc.) to schedule this program to run daily.

### <span style='color:#ff5f27'> 📝 Imports

In [56]:
import datetime
import time
import requests
import pandas as pd
import hopsworks
from mlfs import config
import json
import warnings
warnings.filterwarnings("ignore")

## <span style='color:#ff5f27'> 💱 Get Currency Pair Configuration from Hopsworks </span>

This retrieves the configuration saved in notebook 1, including the currency pair and countries.


In [71]:
project = hopsworks.login()
fs = project.get_feature_store() 
secrets = hopsworks.get_secrets_api()

# Get configuration from Hopsworks secrets
config_str = secrets.get_secret("EXCHANGE_RATE_CONFIG").value
config_dict = json.loads(config_str)

currency_pair = config_dict['currency_pair']
base_currency = config_dict['base_currency']
quote_currency = config_dict['quote_currency']
countries = config_dict['countries']

today = datetime.date.today()

print(f"Currency Pair: {currency_pair}")
print(f"Base Currency: {base_currency}")
print(f"Quote Currency: {quote_currency}")
print(f"Countries: {countries}")
print(f"Today's Date: {today}")

2025-12-17 17:03:42,809 INFO: Closing external client and cleaning up certificates.
Connection closed.
2025-12-17 17:03:42,813 INFO: Initializing external client
2025-12-17 17:03:42,813 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-12-17 17:03:45,246 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1286353
Currency Pair: EUR_SEK
Base Currency: EUR
Quote Currency: SEK
Countries: ['EU', 'Sweden']
Today's Date: 2025-12-17


### <span style="color:#ff5f27;"> 🔮 Get references to the Feature Groups </span>

In [72]:
# Retrieve feature groups
exchange_rate_fg = fs.get_feature_group(
    name='exchange_rate',
    version=1,
)
inflation_fg = fs.get_feature_group(
    
    name='inflation',
    version=2,
)
interest_rate_fg = fs.get_feature_group(
    name='interest_rate',
    version=3,
)

---

## <span style='color:#ff5f27'> Get today's exchnage rate</span>


In [59]:
base_currency = "EUR"
target_currency = "SEK"

url = f"https://api.frankfurter.dev/v1/latest?from={base_currency}&to={target_currency}"
response = requests.get(url)
response.raise_for_status()
data = response.json()

# Build dataframe
today_date = pd.to_datetime(data["date"])

df_new_exchange = pd.DataFrame({
    "date": [today_date],
    "eur_sek_rate": [data["rates"][target_currency]]
})

print("✓ Fetched today's exchange rate")
print(df_new_exchange)

✓ Fetched today's exchange rate
        date  eur_sek_rate
0 2025-12-16        10.942


In [60]:
df_new_exchange.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          1 non-null      datetime64[ns]
 1   eur_sek_rate  1 non-null      float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 144.0 bytes


## <span style='color:#ff5f27'> 🌦 Get inflation rate data (the most recent)</span>

In [61]:
import io
EUROSTAT_URL = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
    "prc_hicp_manr/M.RCH_A.CP00.EU27_2020"
)

PARAMS = {
    "lastNObservations": 1,
    "format": "SDMX-CSV"
}

print("Fetching latest EU inflation from Eurostat...")

response = requests.get(EUROSTAT_URL, params=PARAMS, timeout=60)
response.raise_for_status()

df_raw = pd.read_csv(io.StringIO(response.text))

df_latest_eu_inflation = (
    df_raw[["TIME_PERIOD", "OBS_VALUE"]]
    .dropna()
    .rename(columns={
        "TIME_PERIOD": "date",
        "OBS_VALUE": "eu_inflation_rate"
    })
)

df_latest_eu_inflation["date"] = pd.to_datetime(
    df_latest_eu_inflation["date"] + "-01"
)

print("✓ Latest available EU inflation")
print(df_latest_eu_inflation)


Fetching latest EU inflation from Eurostat...
✓ Latest available EU inflation
        date  eu_inflation_rate
0 2025-10-01                2.5


In [62]:
df_latest_eu_inflation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               1 non-null      datetime64[ns]
 1   eu_inflation_rate  1 non-null      float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 144.0 bytes


## <span style='color:#ff5f27'> 🌦 Get inflation rate data (the most recent)</span>

In [63]:
EUROSTAT_URL = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
    "prc_hicp_manr/M.RCH_A.CP00.SE"
)

PARAMS = {
    "lastNObservations": 1,
    "format": "SDMX-CSV"
}

print("Fetching latest Sweden inflation from Eurostat...")

response = requests.get(EUROSTAT_URL, params=PARAMS, timeout=60)
response.raise_for_status()

df_raw = pd.read_csv(io.StringIO(response.text))

df_latest_sweden_inflation = (
    df_raw[["TIME_PERIOD", "OBS_VALUE"]]
    .dropna()
    .rename(columns={
        "TIME_PERIOD": "date",
        "OBS_VALUE": "sweden_inflation_rate"
    })
)

# Convert YYYY-MM → datetime
df_latest_sweden_inflation["date"] = pd.to_datetime(
    df_latest_sweden_inflation["date"] + "-01"
)

print("✓ Latest available Sweden inflation")
print(df_latest_sweden_inflation)

Fetching latest Sweden inflation from Eurostat...
✓ Latest available Sweden inflation
        date  sweden_inflation_rate
0 2025-10-01                    3.1


## <span style='color:#ff5f27'> 🌦 Get sweden interest rate data (the most recent)</span>

In [64]:

url = "https://api.riksbank.se/swestr/v1/SWESTR"

today = datetime.date.today()
from_date = (today - datetime.timedelta(days=10)).isoformat()

params = {
    "fromDate": f"{from_date}T00:00:00Z",
    "toDate": f"{today.isoformat()}T00:00:00Z"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

df_sweden_interest = pd.DataFrame(data)
df_sweden_interest["date"] = pd.to_datetime(df_sweden_interest["date"])
df_sweden_interest["rate"] = df_sweden_interest["rate"].astype(float)


df_today_sweden_interest = (
    df_sweden_interest
    .sort_values("date")
    .tail(1)
    .reset_index(drop=True)
)

print("✓ Latest available Sweden interest rate (SWESTR)")
print(df_today_sweden_interest)

✓ Latest available Sweden interest rate (SWESTR)
    rate       date  pctl12_5  pctl87_5  volume  alternativeCalculation  \
0  1.632 2025-12-16      1.55      1.75   63201                   False   

  alternativeCalculationReason       publicationTime  republication  \
0                         None  2025-12-17T08:00:00Z          False   

   numberOfTransactions  numberOfAgents  
0                   225               5  


## <span style="color:#ff5f27;">⬆️ Uploading new data to the Feature Store</span>

In [ ]:
df_exchange_rate_clean = df_new_exchange.copy()
df_exchange_rate_clean.rename(columns={'Date': 'date', 'SEK': 'eur_sek_rate'}, inplace=True)
df_exchange_rate_clean['date'] = pd.to_datetime(df_exchange_rate_clean['date'])
df_exchange_rate_clean['eur_sek_rate'] = df_exchange_rate_clean['eur_sek_rate'].astype('float32')
df_exchange_rate_clean = df_exchange_rate_clean[['date', 'eur_sek_rate']]

exchange_rate_fg.insert(df_exchange_rate_clean)
print("✓ Inserted today's exchange rate")



2025-12-17 16:54:03,378 INFO: 	1 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1286353/fs/1273974/fg/1868111


Uploading Dataframe: 100.00% |██████████| Rows 1/1 | Elapsed Time: 00:02 | Remaining Time: 00:00


Launching job: exchange_rate_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1286353/jobs/named/exchange_rate_1_offline_fg_materialization/executions
✓ Inserted today's exchange rate


In [66]:
# Ensure datetime
df_eu_inflation_clean = df_latest_eu_inflation.copy()
df_eu_inflation_clean["date"] = pd.to_datetime(df_eu_inflation_clean["date"])
df_eu_inflation_clean["eu_inflation_rate"] = (
    df_eu_inflation_clean["eu_inflation_rate"].astype("float32")
)
df_eu_inflation_clean = df_eu_inflation_clean[["date", "eu_inflation_rate"]]

df_sweden_inflation_clean = df_latest_sweden_inflation.copy()
df_sweden_inflation_clean["date"] = pd.to_datetime(df_sweden_inflation_clean["date"])
df_sweden_inflation_clean["sweden_inflation_rate"] = (
    df_sweden_inflation_clean["sweden_inflation_rate"].astype("float32")
)
df_sweden_inflation_clean = df_sweden_inflation_clean[
    ["date", "sweden_inflation_rate"]
]

# Merge EU + Sweden inflation on date
df_inflation_clean = df_eu_inflation_clean.merge(
    df_sweden_inflation_clean,
    on="date",
    how="outer"
).sort_values("date").reset_index(drop=True)

# insert inflation data
inflation_fg.insert(df_inflation_clean)
print("✓ Inserted today's inflation rate")

# insert sweden interest rate data

2025-12-17 16:54:24,414 INFO: 	1 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1286353/fs/1273974/fg/1868118


Uploading Dataframe: 100.00% |██████████| Rows 1/1 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: inflation_2_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1286353/jobs/named/inflation_2_offline_fg_materialization/executions
✓ Inserted today's inflation rate


In [73]:

df_interest_clean = df_today_sweden_interest.copy()

df_interest_clean["date"] = pd.to_datetime(df_interest_clean["date"])

df_interest_clean = (
    df_interest_clean
        .rename(columns={"rate": "sweden_interest_rate"})
        .astype({"sweden_interest_rate": "float32"})
        [["date", "sweden_interest_rate"]]
        .sort_values("date")
        .reset_index(drop=True)
)


interest_rate_fg.insert(df_interest_clean)
print("✓ Inserted today's interest rate")


2025-12-17 17:03:54,138 INFO: 	1 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1286353/fs/1273974/fg/1868120


Uploading Dataframe: 100.00% |██████████| Rows 1/1 | Elapsed Time: 00:01 | Remaining Time: 00:00


Launching job: interest_rate_3_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1286353/jobs/named/interest_rate_3_offline_fg_materialization/executions
✓ Inserted today's interest rate


## <span style="color:#ff5f27;">⏭️ **Next:** Part 03: Training Pipeline
 </span> 

In the following notebook you will read from feature groups and create training dataset within the feature store

## <span style="color:#ff5f27;">⏭️ **Next:** Part 03: Training Pipeline
 </span> 

In the following notebook you will read from a feature group and create training dataset within the feature store
